<a href="https://colab.research.google.com/github/Lucio2108/ENTREGA-API/blob/main/entrega_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalamos python-dotenv para gestionar las variables de entorno
!pip install -q python-dotenv requests pandas

import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv

# ==========================================
# 1. CREACIÓN DE ARCHIVOS DE CONFIGURACIÓN (.env y .gitignore)
# ==========================================
# Creamos el archivo .env local (que contiene la URL y token de configuración)
with open(".env", "w") as f:
    f.write("API_URL=https://api.openf1.org/v1/drivers\n")
    f.write("API_KEY=token_seguro_simulacion_2026\n")
    f.write("SESSION_KEY=9158\n")

# Creamos un archivo .env.example (plantilla segura para subir a GitHub)
with open(".env.example", "w") as f:
    f.write("API_URL=https://api.openf1.org/v1/drivers\n")
    f.write("API_KEY=COLOCAR_AQUI_TU_API_KEY\n")
    f.write("SESSION_KEY=9158\n")

# Creamos el archivo .gitignore para bloquear la subida del .env real
with open(".gitignore", "w") as f:
    f.write("# Variables de entorno sensibles\n")
    f.write(".env\n\n")
    f.write("# Archivos temporales de Python\n")
    f.write("__pycache__/\n")
    f.write("*.pyc\n")

# ==========================================
# 2. CARGA SEGURA DE VARIABLES DE ENTORNO
# ==========================================
load_dotenv()

API_URL = os.getenv("API_URL", "https://api.openf1.org/v1/drivers")
API_KEY = os.getenv("API_KEY")
SESSION_KEY = os.getenv("SESSION_KEY", "9158")

# ==========================================
# 3. EXTRACCIÓN VÍA API CON MANEJO DE ERRORES
# ==========================================
print("--- Iniciando petición GET a la API ---")

parametros = {"session_key": SESSION_KEY}
headers = {"Accept": "application/json"}

datos_crudos = []

try:
    respuesta = requests.get(API_URL, params=parametros, headers=headers, timeout=10)

    # Validación explícita de códigos de estado HTTP (200 OK, 404, 500, etc.)
    if respuesta.status_code == 200:
        print(f"✅ Conexión exitosa (Status Code: {respuesta.status_code})")
        datos_crudos = respuesta.json()
    elif respuesta.status_code == 404:
        print("❌ Error 404: El recurso solicitado no fue encontrado en la API.")
    elif respuesta.status_code >= 500:
        print(f"❌ Error del servidor ({respuesta.status_code}): La API externa no está disponible.")
    else:
        print(f"⚠️ Respuesta inesperada. Código HTTP: {respuesta.status_code}")

except requests.exceptions.Timeout:
    print("❌ Error de Timeout: La petición superó el tiempo máximo de espera.")
except requests.exceptions.ConnectionError:
    print("❌ Error de Conexión: Verifica tu conexión a internet o el estado del servidor.")
except requests.exceptions.RequestException as error_general:
    print(f"❌ Error inesperado al consultar la API: {error_general}")

# ==========================================
# 4. TRANSFORMACIÓN BÁSICA Y PERSISTENCIA
# ==========================================
if datos_crudos:
    # Convertimos el JSON a DataFrame y filtramos solo los campos relevantes
    df = pd.DataFrame(datos_crudos)

    columnas_deseadas = [
        "driver_number", "full_name", "name_acronym",
        "team_name", "country_code", "session_key"
    ]

    # Filtramos columnas existentes y eliminamos duplicados o nulos críticos
    columnas_presentes = [col for col in columnas_deseadas if col in df.columns]
    df_limpio = df[columnas_presentes].dropna(subset=["driver_number", "full_name"]).drop_duplicates()

    # Exportamos en ambos formatos requeridos por la consigna
    df_limpio.to_csv("data_extracted.csv", index=False, encoding="utf-8")
    df_limpio.to_json("data_extracted.json", orient="records", indent=4, force_ascii=False)

    print(f"\n✅ Se procesaron {len(df_limpio)} registros correctamente.")
    print("✅ Archivos 'data_extracted.csv' y 'data_extracted.json' guardados con éxito.")
    display(df_limpio.head(10))
else:
    print("⚠️ No se generaron archivos de salida porque no se recuperaron datos.")

--- Iniciando petición GET a la API ---
✅ Conexión exitosa (Status Code: 200)

✅ Se procesaron 20 registros correctamente.
✅ Archivos 'data_extracted.csv' y 'data_extracted.json' guardados con éxito.


,driver_number,full_name,name_acronym,team_name,country_code,session_key
0,1,Max VERSTAPPEN,VER,Red Bull Racing,NED,9158
1,2,Logan SARGEANT,SAR,Williams,USA,9158
2,4,Lando NORRIS,NOR,McLaren,GBR,9158
3,10,Pierre GASLY,GAS,Alpine,FRA,9158
4,11,Sergio PEREZ,PER,Red Bull Racing,MEX,9158
5,14,Fernando ALONSO,ALO,Aston Martin,ESP,9158
6,16,Charles LECLERC,LEC,Ferrari,MON,9158
7,18,Lance STROLL,STR,Aston Martin,CAN,9158
8,20,Kevin MAGNUSSEN,MAG,Haas F1 Team,DEN,9158
9,22,Yuki TSUNODA,TSU,AlphaTauri,JPN,9158
